# Suspended rotary pendulum: 1 s arm-acceleration pulse

Uniform bar, low-speed linear model, with the suspended equilibrium defined as `theta = 0`:

\[
\ddot\theta + a\theta = -b u,\qquad
u=\ddot\phi,
\]

\[
a=\frac{3g}{2l},\qquad b=\frac{3r}{2l}.
\]

This notebook uses \(l=0.235\,\mathrm m\), \(r=0.14\,\mathrm m\), and applies \(u=u_0\) for 1 second followed by \(u=0\).

In [1]:
from pathlib import Path
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Parameters
g = 9.81
l = 0.235   # m, total pendulum-bar length
r = 0.14    # m, rotary-arm radius
a = 3*g/(2*l)
b = 3*r/(2*l)
omega_n = np.sqrt(a)

print(f'a = 3g/(2l) = {a:.6f} 1/s^2')
print(f'b = 3r/(2l) = {b:.6f}')
print(f'natural frequency = {omega_n:.6f} rad/s = {omega_n/(2*np.pi):.6f} Hz')
print(f'Theta(s)/U(s) = -{b:.6f} / (s^2 + {a:.6f})')

a = 3g/(2l) = 62.617021 1/s^2
b = 3r/(2l) = 0.893617
natural frequency = 7.913092 rad/s = 1.259408 Hz
Theta(s)/U(s) = -0.893617 / (s^2 + 62.617021)


## Requested input

The default is \(u_0=1\,\mathrm{rad/s^2}\):

\[
u(t)=\begin{cases}
u_0,&0\le t<1\\0,&t\ge1.\end{cases}
\]

Because \(u=\ddot\phi\), the arm velocity accumulates during the first second. Returning **acceleration** to zero does not return **velocity** to zero.

In [2]:
u0 = 1.0       # rad/s^2
pulse_time = 1.0 # s
t_end = 5.0      # s

def command(t):
    return u0 if t < pulse_time else 0.0

# State x = [theta, theta_dot, phi, phi_dot]
def rhs(t, x):
    theta, theta_dot, phi, phi_dot = x
    u = command(t)
    theta_ddot = -a*theta - b*u
    return [theta_dot, theta_ddot, phi_dot, u]

t_eval = np.linspace(0, t_end, 5001)
sol = solve_ivp(rhs, (0, t_end), [0,0,0,0], t_eval=t_eval,
                max_step=0.002, rtol=1e-9, atol=1e-11)

t = sol.t
theta, theta_dot, phi, phi_dot = sol.y
u = np.where(t < pulse_time, u0, 0.0)

print(f'max |theta| = {np.rad2deg(np.max(np.abs(theta))):.4f} deg')
print(f'phi_dot just after 1 s ≈ {np.interp(1.001, t, phi_dot):.4f} rad/s')
print(f'final phi_dot = {phi_dot[-1]:.4f} rad/s (ideal model keeps moving)')
print(f'final phi = {np.rad2deg(phi[-1]):.2f} deg')

max |theta| = 1.6354 deg
phi_dot just after 1 s ≈ 1.0000 rad/s
final phi_dot = 1.0000 rad/s (ideal model keeps moving)
final phi = 257.83 deg


In [3]:
fig, axes = plt.subplots(4, 1, figsize=(9, 9), sharex=True)
axes[0].plot(t, u)
axes[0].set_ylabel('u = phi_ddot\n[rad/s^2]')
axes[0].grid(True)

axes[1].plot(t, np.rad2deg(theta))
axes[1].set_ylabel('theta [deg]')
axes[1].grid(True)

axes[2].plot(t, phi_dot)
axes[2].set_ylabel('phi_dot [rad/s]')
axes[2].grid(True)

axes[3].plot(t, np.rad2deg(phi))
axes[3].set_ylabel('phi [deg]')
axes[3].set_xlabel('time [s]')
axes[3].grid(True)

fig.suptitle('Suspended pendulum response to 1 s arm angular-acceleration pulse')
fig.tight_layout()
fig_path = Path('figures/suspended_acceleration_pulse_response.png')
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'saved: {fig_path}')

saved: figures/suspended_acceleration_pulse_response.png


![Suspended acceleration pulse response](figures/suspended_acceleration_pulse_response.png)

## Interpretation

- During the 1 s pulse, arm acceleration creates a coupling torque on the pendulum.
- After the pulse, the ideal suspended model becomes a free undamped oscillator, so `theta` keeps oscillating.
- Since the commanded acceleration becomes zero rather than negative, `phi_dot` remains approximately constant after 1 s.
- If the physical experiment must stop the arm, use a deceleration command (for example a symmetric positive/negative acceleration profile) or close a velocity/position loop.